# GP-Hedge on BBOB: how much of the switching headroom does an online portfolio recover?

## Description

*Written 2026-11-10 ~18:30, before the first run. Baseline for the learned selector (issue #11, PR #12).*

**Question.** `bbob_switching` (notebook 04) found large luck-corrected headroom for switching
acquisition functions within a run on noise-free BBOB. The privileged rollout controller beats the tuned
static arm B\* by +59% / +19% (GP, d = 2 / 5) and +66% / +29% (PFN). A controller that uses task
knowledge is a ceiling, not a method.

The standard non-learned way to switch is GP-Hedge (Brochu, Hoffman & de Freitas 2011): an online
portfolio over acquisition functions, rewarded by the updated posterior mean at each arm's nominee.
*How much of the headroom does GP-Hedge recover, and is it better than the tuned static arm at all?*
The answer sets the bar a learned selector has to clear: max(B\*, GP-Hedge).

**Setup.** Everything is identical to the `bbob_switching` confirm sweeps:
- split 1: BBOB f1–f24, instances 2–5, 2 repetitions, i.e. 192 tasks per d;
- d ∈ {2, 5}, n_init = 2d, T = 10d;
- arms ei, pi0.01, ucb0.9, ucb0.99, mes, mean;
- the same candidate sets (CRN keyed by task, step and replicate), the same rank-transformed surrogates;
- f − f_opt, log10(regret + 1e-8).

Each hedge run therefore pairs task by task with:
- the static arms and B\* from `results/bbob_switching/confirm_20261022-145708` (GP, B\* = ucb0.9) and
  `confirm_20261023-075829` (PFN, B\* = ucb0.99);
- the privileged controller and the equal-arms null in the same snapshots.

GP-Hedge, as implemented (`bo/runner.py::run_hedge`, Algorithm 2 of the paper):
- at every step, each arm nominates its maximiser on the shared candidate set;
- one nominee is evaluated, drawn with p ∝ exp(η · g);
- after the posterior update, every arm's gain g is increased by the reward −μ(its nominee), with μ on
  the [0, 1] scale: Φ(μ_z) for the GP, the rank-scale mean for the PFN.

The paper gives no η. Members: η = 1.0 (scikit-optimize's default) and η = 10. Sweep: surrogate ∈ {gp,
pfn} × d ∈ {2, 5} × η ∈ {1, 10}, 8 members, on CPU. A timing probe (24 tasks, d = 5) took 99 s (GP)
and 373 s (PFN).

**What to look for.**
- **Primary:** per surrogate and d, the paired gain of hedge (η = 1) over B\*, as 1 − geometric-mean
  ratio of final regrets with a 95% bootstrap CI over tasks (`analysis.bo.paired_gain`, as in notebook 04).
  Next to it: the privileged gain (ceiling) and the null's gain (noise floor).
- **Secondary:**
  - hedge vs ei (the untuned default, the fair comparison for a user who did not tune B\*);
  - η = 10 vs η = 1;
  - per BBOB function group;
  - arm probabilities over the run (which arms Hedge converges to, and whether that is B\*).

**Reading (pre-registered; this is a baseline measurement, not a gate).**
- CI above 0: hedge recovers headroom. Report the share of the privileged gain, gain_hedge / gain_priv.
- CI below 0: hedge loses to the tuned static arm. The learned selector's bar is then B\*.
- CI includes 0: no detectable difference at n = 192. The null's CI width is the reference for what is
  detectable.
- A **check that the setup is right:** a hedge run whose probabilities stay uniform (|p − 1/6| < 0.02 at
  the last step in more than 90% of tasks) means the rewards carry no signal at this η. That would be
  flagged and investigated before reading the gains.

**Limits.**
- Noise-free BBOB only; the noisy setting (#9) is not covered.
- One seed, one nuisance stream.
- The GP-Hedge reward is computed on each surrogate's own transformed scale. The PFN variant is our
  adaptation, not the paper's method; the GP variant is the paper's.
- B\* was selected on split 0, so it is a tuned baseline that hedge does not get.
- The hedge runs use the CPU, and the confirm sweeps used ulysses' GPU. Tasks and initial designs are the same,
  but the arithmetic is not bit-identical, so the pairing is by task, not by trajectory.


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/bbob_hedge/<sweep_id>"  # set after `just snapshot runs/bbob_hedge/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.